# **Frank Ziegler Programming Assignment 1**

Dataset: https://www.kaggle.com/datasets/laotse/credit-card-approval

# Setup

In [ ]:
from google.colab import drive
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
os.chdir('drive/My Drive/Introduction to Data Mining/Frank Ziegler Programming Assignment 3')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('credit_card_approval.csv')
print(df.head())
df.shape
print(df.isnull().sum)

        ID CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY CNT_CHILDREN  \
0  5065438           F            Y               N  2+ children   
1  5142753           F            N               N  No children   
2  5111146           M            Y               Y  No children   
3  5010310           F            Y               Y   1 children   
4  5010835           M            Y               Y  2+ children   

   AMT_INCOME_TOTAL            NAME_EDUCATION_TYPE    NAME_FAMILY_STATUS  \
0          270000.0  Secondary / secondary special               Married   
1           81000.0  Secondary / secondary special  Single / not married   
2          270000.0               Higher education               Married   
3          112500.0  Secondary / secondary special               Married   
4          139500.0  Secondary / secondary special               Married   

   NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  FLAG_MOBIL  FLAG_WORK_PHONE  \
0       With parents      -13258          -2300       

# Feature Selection & Cleaning

In [ ]:
df_cln = df.copy()

# remove dup entries
df_cln = df_cln.drop_duplicates()

# remove generally redundant or useless columns
# i.e. why do we need "housing type" if we have "owns property"
df_cln = df_cln.drop(columns=['ID', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'STATUS', 'NAME_HOUSING_TYPE'])

# NO NULL VALUES IN THIS DATASET
# I KNOW I KNOW IT'S KINDA CLEAN BUT IT'S LATE AND I CAN'T FIND A BETTER DATASET TO WORK WITH RN

# convert DAYS_BIRTH to AGE and normalize negative values
df_cln['AGE'] = abs(df_cln['DAYS_BIRTH']) // 365
df_cln = df_cln.drop(columns=['DAYS_BIRTH'])
df_cln['DAYS_EMPLOYED'] = abs(df_cln['DAYS_EMPLOYED'])
df_cln['BEGIN_MONTHS'] = abs(df_cln['BEGIN_MONTHS'])

In [ ]:
df_cln.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,DAYS_EMPLOYED,JOB,BEGIN_MONTHS,TARGET,AGE
0,F,Y,N,2+ children,270000.0,Secondary / secondary special,Married,2300,Managers,6,0,36
1,F,N,N,No children,81000.0,Secondary / secondary special,Single / not married,377,Private service staff,4,0,48
2,M,Y,Y,No children,270000.0,Higher education,Married,1028,Laborers,0,0,53
3,F,Y,Y,1 children,112500.0,Secondary / secondary special,Married,1956,Core staff,3,0,41
4,M,Y,Y,2+ children,139500.0,Secondary / secondary special,Married,5578,Drivers,29,0,47


# Outlier Removal using $\text{IQR}$

In [ ]:
# func to remove outliers using IQR
def remove_outliers(df_cln, col):
  q1 = df_cln[col].quantile(0.25)
  q3 = df_cln[col].quantile(0.75)
  iqr = q3 - q1
  return df_cln[(df_cln[col] >= q1 - 1.5 * iqr) & (df_cln[col] <= q3 + 1.5 * iqr)]

# remove income, age, and days employed outliers
df_cln = remove_outliers(df_cln, 'AMT_INCOME_TOTAL')
df_cln = remove_outliers(df_cln, 'DAYS_EMPLOYED')
df_cln = remove_outliers(df_cln, 'AGE')

df_cln.shape

(482994, 12)

# MinMax Normalization

In [ ]:
scaler = MinMaxScaler()
features_to_norm = ['AMT_INCOME_TOTAL', 'DAYS_EMPLOYED']
df_cln[features_to_norm] = scaler.fit_transform(df_cln[features_to_norm])

In [ ]:
df_cln.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,DAYS_EMPLOYED,JOB,BEGIN_MONTHS,TARGET,AGE
0,F,Y,N,2+ children,0.710526,Secondary / secondary special,Married,0.302424,Managers,6,0,36
1,F,N,N,No children,0.157895,Secondary / secondary special,Single / not married,0.047688,Private service staff,4,0,48
2,M,Y,Y,No children,0.710526,Higher education,Married,0.133925,Laborers,0,0,53
3,F,Y,Y,1 children,0.250000,Secondary / secondary special,Married,0.256855,Core staff,3,0,41
4,M,Y,Y,2+ children,0.328947,Secondary / secondary special,Married,0.736654,Drivers,29,0,47


# Convert Categorical Data to Numeric

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df_cln['CODE_GENDER'] = encoder.fit_transform(df_cln['CODE_GENDER'])
df_cln['FLAG_OWN_CAR'] = encoder.fit_transform(df_cln['FLAG_OWN_CAR'])
df_cln['FLAG_OWN_REALTY'] = encoder.fit_transform(df_cln['FLAG_OWN_REALTY'])
df_cln['CNT_CHILDREN'] = encoder.fit_transform(df_cln['CNT_CHILDREN'])
df_cln['NAME_EDUCATION_TYPE'] = encoder.fit_transform(df_cln['NAME_EDUCATION_TYPE'])
df_cln['NAME_FAMILY_STATUS'] = encoder.fit_transform(df_cln['NAME_FAMILY_STATUS'])
df_cln['JOB'] = encoder.fit_transform(df_cln['JOB'])

df_cln.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,DAYS_EMPLOYED,JOB,BEGIN_MONTHS,TARGET,AGE
0,0,1,0,1,0.710526,4,1,0.302424,10,6,0,36
1,0,0,0,2,0.157895,4,3,0.047688,12,4,0,48
2,1,1,1,2,0.710526,1,1,0.133925,8,0,0,53
3,0,1,1,0,0.250000,4,1,0.256855,3,3,0,41
4,1,1,1,1,0.328947,4,1,0.736654,4,29,0,47


# Sampling Data

>Notice that there is a massive class imbalance in the 'TARGET' column:

In [ ]:
df_cln['TARGET'].value_counts()

,count
TARGET,
0,481264
1,1730


> So it's necessary that we majorly sample the data so that our models don't overfit.

***NOTE, YES I KNOW I'M LOSING ALMOST 480,000 ROWS, BUT, I DUNNO WHAT ELSE TO DO LOWKEY***

In [ ]:
# split and sample maj class
majority = df_cln[df_cln['TARGET'] == 0]
minority = df_cln[df_cln['TARGET'] == 1]
majority_sampled = majority.sample(n=len(minority), random_state=42)

# recombine
df_cln = pd.concat([minority, majority_sampled])

# reshuffle
df_cln = df_cln.sample(frac=1, random_state=42).reset_index(drop=True)

df_cln['TARGET'].value_counts()

,count
TARGET,
1,1730
0,1730


# Splitting Data

In [ ]:
from sklearn.model_selection import train_test_split

# predictors
p = df_cln.drop('TARGET', axis = 1)

# target
t = df_cln['TARGET']

# 80-20 train-test split
p_train, p_test, t_train, t_test = train_test_split(p, t, random_state=42, test_size=0.2, shuffle=True)

# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

K = []
training = []
test = []
scores = {}

for k in range(2, 21):
    knn = KNeighborsClassifier(n_neighbors = k)
    knn.fit(p_train, t_train)

    training_score = knn.score(p_train, t_train)
    test_score = knn.score(p_test, t_test)
    K.append(k)

    training.append(training_score)
    test.append(test_score)
    scores[k] = [training_score, test_score]

    print(f'K: {k} | Training Score: {training_score} | Test Score: {test_score}')

K: 2 | Training Score: 0.9436416184971098 | Test Score: 0.7789017341040463
K: 3 | Training Score: 0.8659682080924855 | Test Score: 0.7210982658959537
K: 4 | Training Score: 0.8638005780346821 | Test Score: 0.7326589595375722
K: 5 | Training Score: 0.8171965317919075 | Test Score: 0.6950867052023122
K: 6 | Training Score: 0.8276734104046243 | Test Score: 0.6950867052023122
K: 7 | Training Score: 0.7933526011560693 | Test Score: 0.6589595375722543
K: 8 | Training Score: 0.7742052023121387 | Test Score: 0.6777456647398844
K: 9 | Training Score: 0.7554190751445087 | Test Score: 0.6445086705202312
K: 10 | Training Score: 0.7539739884393064 | Test Score: 0.6546242774566474
K: 11 | Training Score: 0.7463872832369942 | Test Score: 0.634393063583815
K: 12 | Training Score: 0.7453034682080925 | Test Score: 0.6459537572254336
K: 13 | Training Score: 0.7348265895953757 | Test Score: 0.6315028901734104
K: 14 | Training Score: 0.7290462427745664 | Test Score: 0.638728323699422
K: 15 | Training Score

# DecisionTrees

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(p_train, t_train)

training_score = dt.score(p_train, t_train)
test_score = dt.score(p_test, t_test)

print(f'Train: {training_score}')
print(f'Test: {test_score}')

Train: 0.9978323699421965
Test: 0.884393063583815


### Best DecisionTree by F1

In [ ]:
from sklearn.model_selection import GridSearchCV

# hyperparameter to fine tune
param_grid = {
    'max_depth': range(1, 10, 1),
    'min_samples_leaf': range(1, 20, 2),
    'min_samples_split': range(2, 20, 2),
    'criterion': ["entropy", "gini"]
}
# decision tree classifier
tree = DecisionTreeClassifier(random_state=42)
# GridSearchCV
grid_search = GridSearchCV(estimator=tree, param_grid=param_grid,
                           cv=5, verbose=True, scoring='f1')

grid_search.fit(p_train, t_train)

# best score and estimator
print("best f1", grid_search.best_score_)
print(grid_search.best_estimator_)

Fitting 5 folds for each of 1620 candidates, totalling 8100 fits
best f1 0.8350441064108927
DecisionTreeClassifier(max_depth=9, min_samples_split=4, random_state=42)


# Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(p_train, t_train)
training_score = nb.score(p_train, t_train)
test_score = nb.score(p_test, t_test)

print( f'Train: {training_score}' )
print( f'Test: {test_score}' )

Train: 0.5924855491329479
Test: 0.5606936416184971


# Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from prettytable import PrettyTable # because i wanna be fancy and i wanted to figure out how to print nice tables
table = PrettyTable(["model", "confusion matrix", "precision", "recall", "F1 score"])

def eval_model(p_test, t_test, model, header):
  t_pred = model.predict(p_test)
  return [
        header, confusion_matrix(t_test, t_pred), f"{precision_score(t_test, t_pred):.4f}", f"{recall_score(t_test, t_pred):.4f}", f"{f1_score(t_test, t_pred):.4f}"
    ]

# best KNN:
knn = KNeighborsClassifier(n_neighbors=6)
knn.fit(p_train, t_train)
# best DecisionTreeClassifier:
dtc = grid_search.best_estimator_

table.add_row(eval_model(p_test, t_test, knn, 'KNN'))
table.add_row([" ", " ", " ", " ", " "])
table.add_row(eval_model(p_test, t_test, dtc, 'DecisionTree'))
table.add_row([" ", " ", " ", " ", " "])
table.add_row(eval_model(p_test, t_test, nb, 'NaiveBayes'))
#woohoo very sexy

print(table)

+--------------+------------------+-----------+--------+----------+
|    model     | confusion matrix | precision | recall | F1 score |
+--------------+------------------+-----------+--------+----------+
|     KNN      |    [[226 127]    |   0.6675  | 0.7522 |  0.7074  |
|              |    [ 84 255]]    |           |        |          |
|              |                  |           |        |          |
| DecisionTree |    [[276  77]    |   0.7902  | 0.8555 |  0.8215  |
|              |    [ 49 290]]    |           |        |          |
|              |                  |           |        |          |
|  NaiveBayes  |    [[215 138]    |   0.5563  | 0.5103 |  0.5323  |
|              |    [166 173]]    |           |        |          |
+--------------+------------------+-----------+--------+----------+
